# Прогноз успешности 3D-печати детали

Простое базовое решение: загрузка данных, первичный анализ, решающее дерево глубины 4, сохранение `submission.csv`.

## 0. Библиотеки

In [ ]:
import sys
from pathlib import Path

import xgboost as xgb
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

SEED = 42

print("python      ", sys.version.split()[0])
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)

ModuleNotFoundError: No module named 'optuna'

## 1. Загрузка данных

Положите `train.csv`, `test.csv` и `sample_submission.csv` в папку с ноутбуком или укажите путь в `DATA_DIR`.

In [5]:
DATA_DIR = Path("data")  # папка с csv-файлами задачи

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:            ", train.shape)
print("test:             ", test.shape)
print("sample_submission:", sample_submission.shape)

train:             (16000, 21)
test:              (4000, 20)
sample_submission: (4000, 2)


In [6]:
train.head()

,id,material,printer_model,bed_surface,filament_color,nozzle_diameter_mm,nozzle_temp_c,bed_temp_c,print_speed_mm_s,layer_height_mm,...,model_mass_g,contact_area_cm2,geometry_complexity,overhang_angle_deg,estimated_time_min,filament_moisture_pct,cooling_percent,ambient_temp_c,hours_since_maintenance,target
0,115886,ABS,Model_A,textured,blue,0.4,245.4,107.4,35.6,0.32,...,66.2,44.2,0.762,46.2,101.6,NaN,0.0,33.2,357.0,1
1,118129,PLA,Model_C,PEI,natural,0.4,197.6,70.3,78.0,0.12,...,82.2,7.3,0.394,19.8,116.3,0.47,48.5,18.9,538.0,1
2,110735,ABS,Model_A,PEI,white,0.4,232.9,111.9,19.2,0.28,...,57.7,76.7,0.329,36.6,90.4,0.65,51.3,20.0,505.0,1
3,116662,ABS,Model_A,NaN,natural,0.4,224.0,101.1,37.5,0.12,...,34.2,44.4,0.628,33.5,141.5,1.02,3.4,20.8,272.0,1
4,117826,PETG,Model_D,PEI,red,0.4,268.1,93.6,71.7,0.16,...,147.7,19.5,0.361,29.2,199.5,0.47,62.5,21.2,79.0,0


Категориальные и числовые признаки. `id` в модель не передаём.

In [7]:
TARGET = "target"
ID = "id"
CAT_FEATURES = ["material", "printer_model", "bed_surface", "filament_color"]
NUM_FEATURES = [c for c in train.columns if c not in CAT_FEATURES + [ID, TARGET]]

assert list(test.columns) == [c for c in train.columns if c != TARGET]

print(f"Категориальные ({len(CAT_FEATURES)}):", CAT_FEATURES)
print(f"Числовые ({len(NUM_FEATURES)}):", NUM_FEATURES)

Категориальные (4): ['material', 'printer_model', 'bed_surface', 'filament_color']
Числовые (15): ['nozzle_diameter_mm', 'nozzle_temp_c', 'bed_temp_c', 'print_speed_mm_s', 'layer_height_mm', 'infill_percent', 'model_mass_g', 'contact_area_cm2', 'geometry_complexity', 'overhang_angle_deg', 'estimated_time_min', 'filament_moisture_pct', 'cooling_percent', 'ambient_temp_c', 'hours_since_maintenance']


## 2. Первичный анализ

### 2.1. Баланс классов

In [8]:
pd.DataFrame({
    "count": train[TARGET].value_counts().sort_index(),
    "share": train[TARGET].value_counts(normalize=True).sort_index().round(3),
})

,count,share
target,,
0,7207,0.45
1,8793,0.55


### 2.2. Пропуски

In [9]:
missing = pd.DataFrame({
    "train, %": train.isna().mean() * 100,
    "test, %": test.isna().mean() * 100,
}).round(2)
missing[missing["train, %"] > 0]

,"train, %","test, %"
ambient_temp_c,2.48,2.58
bed_surface,2.00,2.00
filament_moisture_pct,3.96,4.15
hours_since_maintenance,1.97,2.12


### 2.3. Описательная статистика числовых признаков

In [10]:
train[NUM_FEATURES].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
nozzle_diameter_mm,16000.0,0.48,0.12,0.40,0.40,0.40,0.60,0.80
nozzle_temp_c,16000.0,228.53,25.70,165.00,210.40,228.80,246.60,295.00
bed_temp_c,16000.0,72.79,25.69,20.00,54.40,71.90,90.80,135.00
print_speed_mm_s,16000.0,56.37,28.65,10.00,35.08,55.80,75.90,155.00
layer_height_mm,16000.0,0.23,0.09,0.10,0.16,0.24,0.28,0.48
infill_percent,16000.0,35.76,24.53,10.00,20.00,30.00,40.00,100.00
model_mass_g,16000.0,117.44,93.88,6.20,56.58,91.10,147.12,800.00
contact_area_cm2,16000.0,39.50,33.04,1.50,18.00,29.80,49.60,260.00
geometry_complexity,16000.0,0.50,0.22,0.02,0.33,0.50,0.67,0.98
overhang_angle_deg,16000.0,33.44,17.17,0.10,19.90,32.20,46.10,78.30


## 3. Подготовка признаков

- пропуски в числовых признаках → медиана;
- пропуски в категориальных признаках → категория `"missing"`;
- категориальные признаки → one-hot.

Предобработка и модель собраны в один `Pipeline`.

In [ ]:
def make_model():
    preprocessor = ColumnTransformer(
        [
            ("num", SimpleImputer(strategy="median"), NUM_FEATURES),
            ("cat", make_pipeline(
                SimpleImputer(strategy="constant", fill_value="missing"),
                OneHotEncoder(handle_unknown="ignore"),
            ), CAT_FEATURES),
        ],
        verbose_feature_names_out=False,
    )
    classifier1 = CatBoostClassifier(
    iterations=3000,  # Максимальное количество деревьев
    learning_rate=0.1,  # Шаг обучения
    depth=6,  # Глубина деревьев
    loss_function='Logloss',  # Функция потерь для бинарной классификации
    eval_metric='AUC',  # Метрика качества на валидации
    random_seed=42,
    verbose=100  # Выводить лог каждые 100 итераций
    )



    return Pipeline([("prep", preprocessor), ("clf", classifier1)])


X = train[CAT_FEATURES + NUM_FEATURES]
y = train[TARGET]
X_test = test[CAT_FEATURES + NUM_FEATURES]

## 4. Модель и валидация

Решающее дерево глубины 4. Для валидации откладываем 20% train.

In [67]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# яндексовская штукня
model = make_model()
model.fit(
    X_train,
    y_train,
)
valid_pred = model.predict(X_valid)

print(f"macro F1 на валидации 1 яндексовские: {f1_score(y_valid, valid_pred, average='macro'):.4f}\n")
print(classification_report(y_valid, valid_pred, digits=3))


0:	total: 7.66ms	remaining: 38.3s
100:	total: 837ms	remaining: 40.6s
200:	total: 1.7s	remaining: 40.5s
300:	total: 2.56s	remaining: 39.9s
400:	total: 3.45s	remaining: 39.6s
500:	total: 4.36s	remaining: 39.2s
600:	total: 5.21s	remaining: 38.1s
700:	total: 6.02s	remaining: 36.9s
800:	total: 6.81s	remaining: 35.7s
900:	total: 7.61s	remaining: 34.6s
1000:	total: 8.45s	remaining: 33.8s
1100:	total: 9.26s	remaining: 32.8s
1200:	total: 10.1s	remaining: 31.8s
1300:	total: 10.9s	remaining: 30.9s
1400:	total: 11.7s	remaining: 30.1s
1500:	total: 12.5s	remaining: 29.1s
1600:	total: 13.3s	remaining: 28.2s
1700:	total: 14.1s	remaining: 27.3s
1800:	total: 14.9s	remaining: 26.5s
1900:	total: 15.8s	remaining: 25.7s
2000:	total: 16.6s	remaining: 24.9s
2100:	total: 17.4s	remaining: 24s
2200:	total: 18.1s	remaining: 23.1s
2300:	total: 18.9s	remaining: 22.2s
2400:	total: 19.7s	remaining: 21.3s
2500:	total: 20.5s	remaining: 20.5s
2600:	total: 21.4s	remaining: 19.7s
2700:	total: 22.2s	remaining: 18.9s
2800:	

Кросс-валидация на 5 фолдах:

In [36]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(make_model(), X, y, cv=cv, scoring="f1_macro")

print("macro F1 по фолдам:", cv_scores.round(4))
print(f"Среднее: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

0:	total: 5.05ms	remaining: 5.04s
100:	total: 395ms	remaining: 3.51s
200:	total: 808ms	remaining: 3.21s
300:	total: 1.24s	remaining: 2.88s
400:	total: 1.66s	remaining: 2.48s
500:	total: 2.1s	remaining: 2.1s
600:	total: 2.55s	remaining: 1.69s
700:	total: 2.94s	remaining: 1.25s
800:	total: 3.35s	remaining: 832ms
900:	total: 3.75s	remaining: 412ms
999:	total: 4.14s	remaining: 0us
0:	total: 4.57ms	remaining: 4.56s
100:	total: 402ms	remaining: 3.58s
200:	total: 822ms	remaining: 3.27s
300:	total: 1.23s	remaining: 2.86s
400:	total: 1.65s	remaining: 2.46s
500:	total: 2.04s	remaining: 2.03s
600:	total: 2.44s	remaining: 1.62s
700:	total: 2.84s	remaining: 1.21s
800:	total: 3.24s	remaining: 805ms
900:	total: 3.63s	remaining: 399ms
999:	total: 4.02s	remaining: 0us
0:	total: 4.71ms	remaining: 4.7s
100:	total: 405ms	remaining: 3.61s
200:	total: 823ms	remaining: 3.27s
300:	total: 1.25s	remaining: 2.9s
400:	total: 1.66s	remaining: 2.47s
500:	total: 2.06s	remaining: 2.05s
600:	total: 2.47s	remaining: 1.

### Важность признаков

Признаки, которые использует дерево (с нулевой важностью не показаны).

In [37]:
feature_names = model.named_steps["prep"].get_feature_names_out()
importances = (
    pd.Series(model.named_steps["clf"].feature_importances_, index=feature_names, name="importance")
    .sort_values(ascending=False)
    .round(4)
)
importances[importances > 0].to_frame()

,importance
nozzle_temp_c,15.4593
bed_temp_c,10.7844
print_speed_mm_s,10.7142
cooling_percent,6.8216
layer_height_mm,5.4928
geometry_complexity,5.3544
overhang_angle_deg,4.7421
model_mass_g,4.4506
filament_moisture_pct,4.3466
material_PLA,4.3056


## 5. Обучение на всём train и предсказание для test

In [38]:
final_model = make_model()
final_model.fit(X, y)
test_pred = final_model.predict(X_test)

0:	total: 5.29ms	remaining: 5.29s
100:	total: 505ms	remaining: 4.5s
200:	total: 969ms	remaining: 3.85s
300:	total: 1.44s	remaining: 3.35s
400:	total: 1.92s	remaining: 2.87s
500:	total: 2.4s	remaining: 2.4s
600:	total: 2.87s	remaining: 1.91s
700:	total: 3.37s	remaining: 1.44s
800:	total: 3.89s	remaining: 967ms
900:	total: 4.43s	remaining: 487ms
999:	total: 4.89s	remaining: 0us


## 6. Сохранение submission

In [39]:
submission = pd.DataFrame({ID: test[ID], TARGET: test_pred.astype(int)})

assert list(submission.columns) == ["id", "target"], "нужны ровно два столбца: id, target"
assert len(submission) == len(test), "число строк должно совпадать с test.csv"
assert submission[ID].is_unique, "id не должны повторяться"
assert set(submission[ID]) == set(sample_submission[ID]), "id должны совпадать с test.csv"
assert submission[TARGET].isin([0, 1]).all(), "target принимает только значения 0 и 1"

submission.to_csv("submission.csv", index=False)
print("Сохранено: submission.csv", submission.shape)
submission.head()

Сохранено: submission.csv (4000, 2)


,id,target
0,118423,1
1,101405,1
2,118315,0
3,105779,0
4,116242,1


## 7. Отправка решения

1. Сохраните ноутбук как скрипт: `jupyter nbconvert --to script baseline.ipynb` (появится `baseline.py`) или через меню `File - Save and Export Notebook As - Executable Script`.
2. Сложите в одну папку `submission.csv`, `baseline.py` и `requirements.txt` (создаётся ячейкой ниже). Ноутбук `.ipynb` и csv-файлы с данными задачи в папку не кладите.
3. Выделите папку с решением (не отдельные файлы).
4. Правой кнопкой → «Отправить» → «Сжатая ZIP-папка» (Windows) или «Сжать» (macOS).
5. Убедитесь, что получившийся `.zip` открывается и внутри видна ваша папка с файлами.
6. Загрузите этот `.zip` на платформу.

In [ ]:
requirements = (
    f"# python {sys.version.split()[0]}\n"
    f"numpy=={np.__version__}\n"
    f"pandas=={pd.__version__}\n"
    f"scikit-learn=={sklearn.__version__}\n"
)
Path("requirements.txt").write_text(requirements, encoding="utf-8")
print(requirements)